# BCS continuo: YOLO26 → grupa → EfficientNet-B0 / Keras → MAE

Notebook autónomo para **Kaggle Notebooks**. Sube `Datasetfotos` como dataset privado con
`train`, `valid` y `test`, cada uno con fotos y `_annotations.coco.json`. No necesita clonar el repositorio.
El código se genera en working; las entradas permanecen de solo lectura.

**Primeras sesiones:** (1) auditoría y preparación; (2) prueba corta del detector, estimación y entrenamiento;
(3) inspección de recortes, prueba corta y entrenamiento del regresor; (4) validación, cierre de decisiones y test.
Puedes combinar etapas si los tiempos medidos lo permiten. No hay garantía de terminar en una sesión.
Consulta en tu cuenta los límites y cuota actuales de Kaggle; no se presupone una duración fija.

Los entrenamientos largos están desactivados. No hay resultados de modelos precalculados en este archivo.
Las pruebas cortas son técnicas, con sus propios checkpoints, y no sustituyen una evaluación.


## 1. Configuración y selección de etapas

Edita solamente esta configuración. El primer recorrido prepara datos sin entrenar.
Para una etapa posterior, desactiva las anteriores y adjunta los outputs persistidos mediante `RESUME_ROOT`.
Elige un nuevo `EXPERIMENT_ID` para cambiar hiperparámetros; un experimento existente protege su configuración.
Al reanudar mantén épocas y parámetros. Los objetivos representan épocas totales por fase.


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys
import time

DATA_ROOT = "/kaggle/input/X"
WORK_ROOT = "/kaggle/working/bcs_project"
RESUME_ROOT = None  # "/kaggle/input/<outputs-previos>/"
EXPERIMENT_ID = "bcs_v1"
SEED = 42

RUN_AUDIT = True
RUN_PREPARE_DATA = True
RUN_TRAIN_DETECTOR = False
RUN_TRAIN_REGRESSOR = False
RUN_EVALUATION = False
RUN_DETECTOR_SMOKE = False
RUN_REGRESSOR_SMOKE = False
RUN_INFERENCE = False
RUN_EXPORT = True

SESSION_BUDGET_MINUTES = None  # Consulta el tiempo disponible mostrado por Kaggle.
SAFETY_MARGIN_MINUTES = 20
SMOKE_EPOCHS = 2
EVALUATION_SPLIT = "val"
FINAL_TEST_CONFIRMED = False  # True solo después de cerrar decisiones usando validación.
REGRESSOR_PHASE = "all"  # "head", "fine_tuning" o "all"
INFERENCE_IMAGE = None  # Ruta a una foto nueva adjunta como entrada.

# Mantener los frameworks de Kaggle. Instalar solo si falta el paquete; sin reemplazar torch/TF.
INSTALL_MISSING_DEPENDENCIES = True
ULTRALYTICS_VERSION = "8.4.0"
BOX_ROUNDING_TOLERANCE_PX = 0.02
DETECTOR = dict(weights="yolo26n.pt", epochs=80, imgsz=640, batch=4,
                device="0", patience=15, save_period=5)
REGRESSOR = dict(weights="imagenet", size=224, batch=4, device="0",
                 head_epochs=30, head_lr=1e-3, fine_tuning_epochs=20,
                 fine_tuning_lr=1e-5, fine_layers=20, patience=8)
INFERENCE = dict(confidence=0.25, association_coverage=0.8)

# También puedes apuntar DETECTOR['weights'] y REGRESSOR['weights'] a pesos locales
# adjuntos a Kaggle para trabajar sin Internet. No usar pesos aleatorios para el experimento final.
assert Path(EXPERIMENT_ID).name == EXPERIMENT_ID and EXPERIMENT_ID not in ("", ".", "..")
assert SESSION_BUDGET_MINUTES is None or SESSION_BUDGET_MINUTES > SAFETY_MARGIN_MINUTES
assert SAFETY_MARGIN_MINUTES >= 0 and SMOKE_EPOCHS >= 2
assert str(DETECTOR['device']).isdigit() and str(REGRESSOR['device']).isdigit()
assert 0 <= INFERENCE['confidence'] <= 1 and 0 < INFERENCE['association_coverage'] <= 1
assert BOX_ROUNDING_TOLERANCE_PX >= 0
assert all(DETECTOR[k] > 0 for k in ('epochs', 'imgsz', 'batch', 'patience', 'save_period'))
assert all(REGRESSOR[k] > 0 for k in ('size', 'batch', 'head_epochs', 'head_lr',
                                    'fine_tuning_epochs', 'fine_tuning_lr', 'fine_layers', 'patience'))
assert REGRESSOR['weights'] is not None, 'Los pesos aleatorios se reservan a la prueba técnica.'
if Path('/kaggle').exists():
    assert Path(WORK_ROOT).resolve().is_relative_to(Path('/kaggle/working')), 'WORK_ROOT debe estar en working.'

# El reloj conserva el inicio al repetir esta celda en el mismo kernel.
_SESSION_START_UNIX = globals().get('_SESSION_START_UNIX', time.time())
DEADLINE_UNIX = (None if SESSION_BUDGET_MINUTES is None else
                 _SESSION_START_UNIX + (SESSION_BUDGET_MINUTES-SAFETY_MARGIN_MINUTES)*60)
Path(WORK_ROOT, 'src').mkdir(parents=True, exist_ok=True)
SOURCE_FILES = {}


### Módulos generados

Estas celdas contienen el código fuente del entregable. Se escriben antes de iniciar procesos;
no cargan frameworks en el kernel. Para mantener el notebook desde el repositorio, editar `kaggle_src`
y ejecutar `python3 build_kaggle_notebook.py`.


**Módulo `common.py`**


In [ ]:
SOURCE_FILES['common.py'] = r'''"""Utilidades sin importar frameworks GPU."""
import csv
import hashlib
import json
import math
import os
import shutil
import time
from pathlib import Path

SCHEMA = 1


def require(value, message):
    if not value:
        raise RuntimeError(message)


def load(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def save(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + '\n', encoding='utf-8')
    os.replace(temp, path)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_csv(path, records, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = fields or list(records[0])
    temp = path.with_name(path.name + '.tmp')
    with temp.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)
    temp.replace(path)


def copy_atomic(source, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_name(target.name + '.tmp')
    shutil.copy2(source, temp)
    temp.replace(target)


def artifacts(cfg):
    return Path(cfg['WORK_ROOT']) / 'artifacts'


def stage(cfg, name, fingerprint, outputs, details=None):
    root = artifacts(cfg)
    save(root / 'state' / (name + '.json'), {
        'schema': SCHEMA, 'status': 'complete', 'fingerprint': fingerprint,
        'outputs': {str(Path(p).relative_to(root)): sha(p) for p in outputs},
        'details': details or {},
    })


def reusable(cfg, name, fingerprint):
    root = artifacts(cfg)
    path = root / 'state' / (name + '.json')
    if not path.exists():
        return False
    state = load(path)
    require(state['fingerprint'] == fingerprint,
            f'Etapa {name} incompatible: cambió código, datos o configuración. Usa otro WORK_ROOT/EXPERIMENT_ID.')
    require(all((root / p).is_file() and sha(root / p) == h for p, h in state['outputs'].items()),
            f'Archivos incompletos o alterados en {name}. Recupera un export íntegro.')
    print('Reutilizar:', name)
    return True


def contract(cfg):
    root = artifacts(cfg)
    manifest = root / 'manifests' / 'manifest.json'
    split = root / 'manifests' / 'splits.json'
    require(manifest.is_file() and split.is_file(), 'Ejecuta auditoría y preparación primero.')
    m = load(manifest)
    s = load(split)
    require(s['dataset_hash'] == m['dataset_hash'], 'Particiones de otra versión de datos.')
    return {'schema': SCHEMA, 'dataset_hash': m['dataset_hash'], 'split_hash': s['split_hash'],
            'code_hash': cfg['CODE_HASH']}


def rows(cfg, split=None, supervised=False):
    data = load(artifacts(cfg) / 'manifests' / 'manifest.json')['rows']
    return [r for r in data if (split is None or r['split'] == split)
            and (not supervised or r['bcs_status'] == 'ok')]


def image_path(cfg, row):
    return Path(cfg['DATA_ROOT']) / row['image']


def crop(image, xyxy, size):
    from PIL import Image, ImageOps
    x1, y1, x2, y2 = xyxy
    box = (max(0, math.floor(x1)), max(0, math.floor(y1)),
           min(image.width, math.ceil(x2)), min(image.height, math.ceil(y2)))
    require(box[2] > box[0] and box[3] > box[1], 'Recorte vacío')
    return ImageOps.pad(image.convert('RGB').crop(box), (size, size),
                        method=Image.Resampling.BILINEAR, color=(0, 0, 0))


def pair_boxes(boxes, coverage=.8):
    pairs = []
    for cow in [b for b in boxes if b['class'] == 0]:
        a, b, c, d = cow['xyxy']
        for rump in [b for b in boxes if b['class'] == 1]:
            x, y, z, w = rump['xyxy']
            area = (z-x)*(w-y)
            inside = max(0, min(c, z)-max(a, x))*max(0, min(d, w)-max(b, y))
            if area > 0 and a <= (x+z)/2 <= c and b <= (y+w)/2 <= d and inside/area >= coverage:
                pairs.append((cow, rump))
    return ('ok', pairs[0]) if len(pairs) == 1 else (
        'sin_par_valido' if not pairs else 'asociacion_ambigua', None)


def remaining(cfg):
    deadline = cfg.get('DEADLINE_UNIX')
    return None if deadline is None else deadline - time.time()


def can_start(cfg, seconds):
    left = remaining(cfg)
    return left is None or left > max(0, seconds) * 1.25


def restore(cfg):
    source = cfg.get('RESUME_ROOT')
    if not source:
        return
    source = Path(source)
    candidates = [source] if (source / 'state').is_dir() else list(source.rglob('artifacts'))
    candidates = [p for p in candidates if (p / 'state').is_dir() and (p / 'configs').is_dir()]
    require(len(candidates) == 1, 'RESUME_ROOT debe contener exactamente una carpeta artifacts.')
    source = candidates[0]
    inventory = load(source / 'inventory.json')
    require(bool(inventory), 'Export sin inventario de integridad.')
    require(all(not Path(p).is_absolute() and '..' not in Path(p).parts for p in inventory),
            'Ruta insegura en inventario.')
    require(all((source / p).is_file() and sha(source / p) == h for p, h in inventory.items()),
            'Export previo incompleto o con hashes incorrectos.')
    marker = Path(cfg['WORK_ROOT']) / 'resume_import.json'
    if marker.exists():
        require(load(marker)['inventory_hash'] == digest(inventory),
                'Ya se importó otro export en este WORK_ROOT; usa una carpeta nueva.')
        print('Este export ya se importó; conservar el progreso actual de working.')
        return
    target = artifacts(cfg)
    for p, h in inventory.items():
        dest = target / p
        if dest.exists():
            require(sha(dest) == h, f'Conflicto al importar {p}; usa un WORK_ROOT vacío.')
        else:
            copy_atomic(source / p, dest)
    save(marker, {'inventory_hash': digest(inventory)})
    print('Artefactos verificados y copiados a working.')
'''
Path(WORK_ROOT, 'src', 'common.py').write_text(SOURCE_FILES['common.py'], encoding='utf-8')


**Módulo `data.py`**


In [ ]:
SOURCE_FILES['data.py'] = r'''"""COCO directo: auditoría, particiones originales y exportaciones reproducibles."""
import collections
import json
import math
import re
from pathlib import Path
from PIL import Image
from common import (artifacts, contract, crop, digest, image_path, load, require,
                    reusable, rows, save, sha, stage, write_csv)


def discover(input_root):
    root = Path(input_root)
    return sorted({p.parent.parent for p in root.rglob('_annotations.coco.json')
                   if p.parent.name == 'train'
                   and (p.parent.parent / 'valid/_annotations.coco.json').is_file()
                   and (p.parent.parent / 'test/_annotations.coco.json').is_file()})


def inspect(cfg):
    source = Path(cfg['DATA_ROOT'])
    for split in ('train', 'valid', 'test'):
        d = load(source / split / '_annotations.coco.json')
        print(split, 'imágenes:', len(d['images']), 'cajas:', len(d['annotations']),
              'categorías:', [(c['id'], c['name']) for c in d['categories']])


def audit(cfg):
    root = artifacts(cfg)
    source = Path(cfg['DATA_ROOT'])
    records, corrections, excluded = [], [], []
    cow_splits, hash_splits = collections.defaultdict(set), collections.defaultdict(set)
    for original, split in [('train', 'train'), ('valid', 'val'), ('test', 'test')]:
        d = load(source / original / '_annotations.coco.json')
        cats = {c['id']: c['name'] for c in d['categories']}
        require(len(cats) == len(d['categories']), 'IDs de categoría duplicados.')
        require({'cow', 'ass'} <= set(cats.values()), 'Faltan categorías cow/ass.')
        images = {im['id']: im for im in d['images']}
        require(len(images) == len(d['images']), 'IDs de imagen duplicados.')
        annotations = collections.defaultdict(list)
        seen_annotations = set()
        for ann in d['annotations']:
            require(ann['id'] not in seen_annotations, 'ID de anotación duplicado.')
            seen_annotations.add(ann['id'])
            require(ann['image_id'] in images and ann['category_id'] in cats, 'Referencia COCO inválida.')
            require(cats[ann['category_id']] in ('cow', 'ass'), 'Categoría anotada desconocida.')
            require(ann.get('iscrowd', 0) == 0, 'iscrowd requiere revisión manual.')
            annotations[ann['image_id']].append(ann)
        seen_names = set()
        for im in d['images']:
            name = im['file_name']
            require(Path(name).name == name and '\\' not in name, 'Nombre de archivo inseguro.')
            require(name not in seen_names, 'Archivo repetido en COCO.')
            seen_names.add(name)
            path = source / original / name
            require(path.is_file(), f'Falta foto: {path}')
            with Image.open(path) as picture:
                require(picture.size == (im['width'], im['height']), 'Dimensiones COCO incorrectas.')
                picture.verify()
            with Image.open(path) as picture:
                picture.convert('RGB').load()
            tags = im.get('extra', {}).get('user_tags', [])
            ids = sorted({str(t) for t in tags if re.fullmatch(r'\d+', str(t))})
            btags = sorted({str(t) for t in tags if str(t).startswith('BCS_')})
            require(ids, 'Falta identidad de vaca: no se puede verificar la partición.')
            for cow in ids:
                cow_splits[cow].add(split)
            bcs, status = None, 'ambiguo_o_ausente'
            if len(ids) == len(btags) == 1:
                match = re.fullmatch(r'BCS_(\d+)(?:_(\d+))?', btags[0])
                if match:
                    bcs = float(match[1] + ('.' + match[2] if match[2] else ''))
                    require(1 <= bcs <= 5, 'BCS fuera de 1..5.')
                    status = 'ok'
            boxes = []
            for ann in annotations[im['id']]:
                x, y, w, h = ann['bbox']
                require(all(math.isfinite(v) for v in (x, y, w, h)) and w > 0 and h > 0,
                        'Caja COCO no finita o vacía.')
                excess = max(0, -x, -y, x+w-im['width'], y+h-im['height'])
                require(excess <= cfg['BOX_ROUNDING_TOLERANCE_PX'] + 1e-9,
                        f'Caja fuera de imagen: {original}/{name}, exceso={excess}')
                xyxy = [max(0, x), max(0, y), min(im['width'], x+w), min(im['height'], y+h)]
                require(xyxy[2] > xyxy[0] and xyxy[3] > xyxy[1], 'Caja vacía al limitar.')
                if excess > 1e-9:
                    corrections.append({'key': f'{original}/{name}', 'annotation_id': ann['id'],
                                        'original_xywh': ann['bbox'], 'corrected_xyxy': xyxy})
                boxes.append({'class': {'cow': 0, 'ass': 1}[cats[ann['category_id']]], 'xyxy': xyxy})
            require(collections.Counter(b['class'] for b in boxes) == {0: 1, 1: 1},
                    'Se espera una caja cow y una ass por foto; revisa el COCO.')
            h = sha(path)
            hash_splits[h].add(split)
            row = {'key': f'{original}/{name}', 'image': f'{original}/{name}', 'split': split,
                   'image_id': im['id'], 'cow_ids': ids, 'cow_id': ids[0] if len(ids) == 1 else None,
                   'bcs': bcs, 'bcs_status': status, 'tags': tags, 'sha256': h,
                   'width': im['width'], 'height': im['height'], 'boxes': boxes}
            records.append(row)
            if status != 'ok':
                excluded.append({'key': row['key'], 'split': split, 'reason': status})
    overlap = {k: sorted(v) for k, v in cow_splits.items() if len(v) > 1}
    duplicates = [h for h, splits in hash_splits.items() if len(splits) > 1]
    issues = {'cross_split_cows': overlap, 'cross_split_hashes': duplicates,
              'box_corrections': corrections, 'excluded_bcs': excluded}
    # No sobrescribir la evidencia de una auditoría válida anterior si falla un intento nuevo.
    if overlap or duplicates:
        save(root / 'reports/audit_failed_attempt.json', issues)
    require(not overlap and not duplicates, 'Fuga entre particiones: revisa reports/audit_failed_attempt.json.')
    records.sort(key=lambda r: r['key'])
    require(all(any(r['split'] == s and r['bcs_status'] == 'ok' for r in records)
                for s in ('train', 'val', 'test')), 'Partición BCS vacía.')
    source_hashes = {s: sha(source / s / '_annotations.coco.json') for s in ('train', 'valid', 'test')}
    dataset_hash = digest({'rows': records, 'coco': source_hashes})
    fp = digest({'dataset': dataset_hash, 'code': cfg['CODE_HASH'],
                 'tolerance': cfg['BOX_ROUNDING_TOLERANCE_PX']})
    if reusable(cfg, 'audit', fp):
        return
    save(root / 'reports/audit_issues.json', issues)
    target = root / 'manifests/manifest.json'
    save(target, {'dataset_hash': dataset_hash, 'coco_hashes': source_hashes, 'rows': records})
    write_csv(root / 'manifests/excluded_bcs.csv', excluded, ['key', 'split', 'reason'])
    totals = [len(records), sum(r['bcs_status'] == 'ok' for r in records)]
    summary = []
    for split in ('train', 'val', 'test'):
        selected = [r for r in records if r['split'] == split]
        eligible = [r for r in selected if r['bcs_status'] == 'ok']
        summary.append({'split': split, 'detector_images': len(selected), 'regressor_images': len(eligible),
                        'excluded_bcs': len(selected)-len(eligible),
                        'detector_pct': 100*len(selected)/totals[0], 'regressor_pct': 100*len(eligible)/totals[1],
                        'detector_cows': len({c for r in selected for c in r['cow_ids']}),
                        'regressor_cows': len({r['cow_id'] for r in eligible}),
                        'bcs_distribution': dict(collections.Counter(str(r['bcs']) for r in eligible))})
    save(root / 'reports/audit_summary.json', summary)
    stage(cfg, 'audit', fp, [target, root / 'reports/audit_summary.json', root / 'reports/audit_issues.json',
                            root / 'manifests/excluded_bcs.csv'])
    print(json.dumps(summary, indent=2))


def partition(cfg):
    root = artifacts(cfg)
    m = load(root / 'manifests/manifest.json')
    assignments = {r['key']: r['split'] for r in m['rows']}
    fp = digest({'dataset': m['dataset_hash'], 'assignments': assignments, 'code': cfg['CODE_HASH']})
    if reusable(cfg, 'partitions', fp):
        return
    path = root / 'manifests/splits.json'
    save(path, {'dataset_hash': m['dataset_hash'], 'split_hash': digest(assignments),
                'policy': 'original_by_cow', 'assignments': assignments})
    stage(cfg, 'partitions', fp, [path])


def export_yolo(cfg):
    root = artifacts(cfg)
    fp = digest(contract(cfg))
    target = root / 'prepared/yolo'
    # Las rutas del YAML se regeneran para la sesión actual.
    yaml = root / 'configs/detector.yaml'
    if not reusable(cfg, 'yolo_export', fp):
        outputs = []
        for r in rows(cfg):
            dest = target / 'images' / r['split'] / (r['sha256'][:16] + '_' + Path(r['image']).name)
            dest.parent.mkdir(parents=True, exist_ok=True)
            # Copias portables: no guardar symlinks hacia /kaggle/input.
            from common import copy_atomic
            copy_atomic(image_path(cfg, r), dest)
            label = target / 'labels' / r['split'] / (dest.stem + '.txt')
            label.parent.mkdir(parents=True, exist_ok=True)
            lines = []
            for b in r['boxes']:
                x, y, z, w = b['xyxy']
                lines.append(f"{b['class']} {(x+z)/2/r['width']:.10f} {(y+w)/2/r['height']:.10f} "
                             f"{(z-x)/r['width']:.10f} {(w-y)/r['height']:.10f}")
            label.write_text('\n'.join(lines)+'\n')
            outputs.extend([dest, label])
        stage(cfg, 'yolo_export', fp, outputs)
    save(yaml, {'path': str(target), 'train': 'images/train', 'val': 'images/val',
                'test': 'images/test', 'names': {0: 'cow', 1: 'ass'}})


def samples(cfg):
    root = artifacts(cfg)
    fp = digest({**contract(cfg), 'size': cfg['REGRESSOR']['size']})
    if reusable(cfg, 'regression_samples', fp):
        return
    output, index = [], []
    for r in rows(cfg, supervised=True):
        path = root / 'prepared/crops_gt' / r['split'] / (digest(r['key']) + '.png')
        path.parent.mkdir(parents=True, exist_ok=True)
        box = next(b['xyxy'] for b in r['boxes'] if b['class'] == 1)
        with Image.open(image_path(cfg, r)) as im:
            crop(im, box, cfg['REGRESSOR']['size']).save(path)
        output.append(path)
        index.append({'key': r['key'], 'split': r['split'], 'cow_id': r['cow_id'], 'bcs': r['bcs'],
                      'crop': str(path.relative_to(root))})
    path = root / 'manifests/regression_samples.json'
    save(path, index)
    stage(cfg, 'regression_samples', fp, output+[path])
'''
Path(WORK_ROOT, 'src', 'data.py').write_text(SOURCE_FILES['data.py'], encoding='utf-8')


**Módulo `detector.py`**


In [ ]:
SOURCE_FILES['detector.py'] = r'''"""YOLO26: checkpoints recuperables antes de strip_optimizer y pausa por época."""
import importlib.metadata
import json
import os
import time
from pathlib import Path
from PIL import Image
from common import (artifacts, can_start, contract, copy_atomic, crop, digest, image_path, load,
                    pair_boxes, require, rows, save, sha, stage)


def runtime(cfg):
    import torch
    import ultralytics
    require(ultralytics.__version__ == cfg['ULTRALYTICS_VERSION'], 'Versión Ultralytics incompatible.')
    require(torch.cuda.is_available(), 'Se requiere GPU CUDA para esta etapa.')
    require(str(cfg['DETECTOR']['device']).isdigit(),
            'Este notebook valida una GPU. Multi-GPU requiere verificar DDP, reanudación y mejora de tiempo.')
    return torch


def run_dir(cfg, smoke=False):
    return artifacts(cfg) / ('smoke/detector' if smoke else 'detector') / cfg['EXPERIMENT_ID']


def publish(cfg, run, state, smoke):
    root = artifacts(cfg)
    history = run / 'ultralytics/results.csv'
    if history.is_file():
        copy_atomic(history, root / 'histories' / ('detector_smoke.csv' if smoke else 'detector.csv'))
    if smoke:
        save(root / 'reports/detector_smoke.json', {
            'contract': contract(cfg), 'settings': cfg['DETECTOR'], 'seed': cfg['SEED'],
            'seconds_per_epoch': state['seconds_per_epoch'],
            'peak_gpu_allocated_bytes': state['peak_gpu_allocated_bytes'],
            'completed_epochs': state['completed_epochs'], 'resume_verified': state.get('resume_verified', False),
            'estimated_full_minutes': state['seconds_per_epoch']*cfg['DETECTOR']['epochs']/60})
    elif state['status'] == 'complete':
        stage(cfg, 'detector_trained', state['fingerprint'],
              [run / 'best.pt', run / 'last.pt', run / 'resume.pt', run / 'state.json'])


def train(cfg, smoke=False, interrupt_after=None):
    torch = runtime(cfg)
    from ultralytics import YOLO
    from ultralytics.engine.trainer import BaseTrainer
    import inspect
    root = artifacts(cfg)
    settings = dict(cfg['DETECTOR'])
    settings['epochs'] = cfg['SMOKE_EPOCHS'] if smoke else settings['epochs']
    settings['save_period'] = 1 if smoke else settings['save_period']
    run = run_dir(cfg, smoke)
    signature = {**contract(cfg), 'settings': settings, 'ultralytics': importlib.metadata.version('ultralytics'),
                 'torch': torch.__version__, 'smoke': smoke, 'seed': cfg['SEED']}
    fp = digest(signature)
    status_file = run / 'state.json'
    previous = load(status_file) if status_file.exists() else None
    if previous:
        require(previous['fingerprint'] == fp, 'Detector incompatible: usa otro EXPERIMENT_ID para cambiar épocas o parámetros.')
        require(sha(run / previous['resume_checkpoint']) == previous['resume_sha256'], 'Checkpoint recuperable incompleto.')
        require(sha(run / previous['best_checkpoint']) == previous['best_sha256'], 'Mejor checkpoint incompleto.')
        copy_atomic(run / previous['resume_checkpoint'], run / 'resume.pt')
        copy_atomic(run / previous['resume_checkpoint'], run / 'last.pt')
        copy_atomic(run / previous['best_checkpoint'], run / 'best.pt')
        if previous['status'] == 'complete':
            require(sha(run / 'best.pt') == previous['best_sha256'], 'best.pt alterado.')
            publish(cfg, run, previous, smoke)
            print('Detector ya completado:', run)
            return
    # Verificar orden de callbacks sobre la versión fijada, además de guardar el código en environment.
    source = inspect.getsource(BaseTrainer._do_train)
    require('on_model_save' in source and 'on_fit_epoch_end' in source, 'Callbacks YOLO incompatibles.')
    estimate_file = root / 'reports/detector_smoke.json'
    if not smoke:
        require(estimate_file.is_file(), 'Ejecuta primero la prueba corta del detector.')
        estimate = load(estimate_file)
        require(estimate['contract'] == contract(cfg) and estimate['settings'] == cfg['DETECTOR']
                and estimate['seed'] == cfg['SEED'],
                'La prueba corta no corresponde a estos datos/configuración.')
        if not can_start(cfg, estimate['seconds_per_epoch']):
            print('Presupuesto insuficiente para iniciar otra época del detector.')
            return
    require(can_start(cfg, 0), 'Se agotó el presupuesto útil de sesión.')
    run.mkdir(parents=True, exist_ok=True)
    save(run / 'signature.json', signature)
    weights = settings['weights']
    if previous:
        checkpoint = run / 'resume.pt'
        require(checkpoint.is_file() and sha(checkpoint) == previous['resume_sha256'], 'Checkpoint recuperable incompleto.')
        # Solo checkpoints del propio proyecto; torch.load usa pickle.
        ckpt = torch.load(checkpoint, map_location='cpu', weights_only=False)
        require(ckpt.get('optimizer') is not None and ckpt.get('epoch', -1) >= 0,
                'Pesos sin estado de entrenamiento: inicia un experimento nuevo desde pesos.')
        require(ckpt['epoch'] + 1 == previous['completed_epochs'], 'Época de checkpoint y estado no coincide.')
        require(ckpt['epoch'] + 1 < settings['epochs'], 'Entrenamiento terminado: usa un experimento nuevo.')
        # Reubicar exclusivamente rutas, sin modificar épocas ni optimizador.
        ckpt['train_args'].update(data=str(root / 'configs/detector.yaml'),
                                  project=str(run), name='ultralytics', save_dir=str(run / 'ultralytics'))
        relocated = run / 'resume_session.pt'
        torch.save(ckpt, relocated)
        del ckpt
        model = YOLO(str(relocated))
    else:
        require(not (run / 'ultralytics').exists(), 'Run incompleto sin estado: usa otro EXPERIMENT_ID.')
        model = YOLO(weights)
    epoch_start = [None]
    durations = list(previous.get('epoch_durations', [])) if previous else []
    paused = [False]
    previous_epochs = previous['completed_epochs'] if previous else 0
    state = dict(previous or {})

    def begin(trainer):
        epoch_start[0] = time.monotonic()

    def checkpoint(trainer):
        if int(os.environ.get('RANK', '-1')) not in (-1, 0):
            return
        elapsed = time.monotonic() - epoch_start[0]
        durations.append(elapsed)
        resume_name = f'resume_epoch_{trainer.epoch+1:04d}.pt'
        best_name = f'best_epoch_{trainer.epoch+1:04d}.pt'
        copy_atomic(trainer.last, run / resume_name)
        copy_atomic(trainer.last, run / 'resume.pt')
        copy_atomic(trainer.last, run / 'last.pt')
        if trainer.best.is_file():
            copy_atomic(trainer.best, run / best_name)
            copy_atomic(trainer.best, run / 'best.pt')
        state.update(fingerprint=fp, stage='detector', status='complete' if trainer.stop else 'running',
                     completed_epochs=trainer.epoch+1, target_epochs=settings['epochs'],
                     resume_checkpoint=resume_name, best_checkpoint=best_name,
                     resume_sha256=sha(run / resume_name), best_sha256=sha(run / best_name),
                     seconds_per_epoch=sum(durations)/len(durations),
                     epoch_durations=durations,
                     resume_verified=previous is not None,
                     best_fitness=float(trainer.best_fitness),
                     peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated(),
                     callback_resume_note='Ultralytics restaura estado del checkpoint; paciencia y RNG no se garantizan idénticos.')
        save(status_file, state)
        for pattern in ('resume_epoch_*.pt', 'best_epoch_*.pt'):
            for old in sorted(run.glob(pattern))[:-2]:
                old.unlink()
        forced = interrupt_after is not None and trainer.epoch+1 >= interrupt_after
        if trainer.epoch+1 < settings['epochs'] and (forced or not can_start(cfg, max(durations))):
            trainer.stop = True
            paused[0] = True

    model.add_callback('on_train_epoch_start', begin)
    model.add_callback('on_model_save', checkpoint)
    options = dict(data=str(root / 'configs/detector.yaml'), epochs=settings['epochs'],
                   imgsz=settings['imgsz'], batch=settings['batch'], device=settings['device'],
                   workers=2, seed=cfg['SEED'], deterministic=True, save=True,
                   save_period=settings['save_period'], patience=settings['patience'],
                   project=str(run), name='ultralytics', exist_ok=True, cache=False, plots=True,
                   amp=True, val=True)
    if previous:
        options['resume'] = True
    model.train(**options)
    require(state.get('completed_epochs', 0) > previous_epochs, 'No se guardó una época completa.')
    state['status'] = 'paused' if paused[0] else 'complete'
    state['best_sha256'] = sha(run / 'best.pt')
    save(status_file, state)
    publish(cfg, run, state, smoke)
    print(json.dumps(state, indent=2))


def trained(cfg):
    run = run_dir(cfg)
    state = load(run / 'state.json')
    require(state['status'] == 'complete', 'Detector pendiente; termina el entrenamiento antes de evaluar.')
    signature = load(run / 'signature.json')
    require(all(signature[k] == v for k, v in contract(cfg).items()), 'Detector de otros datos/código.')
    require(signature['settings'] == cfg['DETECTOR'], 'Configuración detector distinta del entrenamiento.')
    require(signature['seed'] == cfg['SEED'], 'Semilla detector distinta del entrenamiento.')
    require(sha(run / 'best.pt') == state['best_sha256'], 'Checkpoint detector alterado.')
    return run / 'best.pt'


def detect(cfg, split=None, external=None):
    runtime(cfg)
    from ultralytics import YOLO
    root = artifacts(cfg)
    weights = trained(cfg)
    model = YOLO(str(weights))
    require(model.names == {0: 'cow', 1: 'ass'}, 'Clases del detector incorrectas.')
    sources = rows(cfg, split, supervised=True) if external is None else [
        {'key': Path(external).name, 'image': external, 'bcs': None, 'cow_id': None}]
    tag = split if external is None else 'external'
    dest = root / 'predicted' / tag
    settings = {'detector_hash': sha(weights), **contract(cfg), 'inference': cfg['INFERENCE'],
                'size': cfg['REGRESSOR']['size']}
    predictions = []
    for row in sources:
        source = image_path(cfg, row) if external is None else Path(external)
        result = model.predict(source=str(source), imgsz=cfg['DETECTOR']['imgsz'],
                               conf=cfg['INFERENCE']['confidence'], device=cfg['DETECTOR']['device'], verbose=False)[0]
        boxes = [{'class': int(c), 'xyxy': [float(v) for v in xy], 'confidence': float(score)}
                 for xy, c, score in zip(result.boxes.xyxy.cpu().tolist(), result.boxes.cls.cpu().tolist(),
                                         result.boxes.conf.cpu().tolist())]
        status, pair = pair_boxes(boxes, cfg['INFERENCE']['association_coverage'])
        dest.mkdir(parents=True, exist_ok=True)
        crop_path = None
        if status == 'ok':
            p = dest / (digest(row['key']) + '.png')
            with Image.open(source) as im:
                crop(im, pair[1]['xyxy'], cfg['REGRESSOR']['size']).save(p)
            crop_path = str(p.relative_to(root))
        # Vista de las primeras nueve imágenes y toda inferencia externa.
        if len(predictions) < 9:
            result.save(filename=str(dest / (digest(row['key']) + '_boxes.jpg')))
        predictions.append({'key': row['key'], 'cow_id': row['cow_id'], 'true_bcs': row['bcs'],
                            'status': status, 'boxes': boxes, 'crop': crop_path})
    save(dest / 'predictions.json', {'settings': settings, 'rows': predictions})


def evaluate(cfg, split):
    runtime(cfg)
    from ultralytics import YOLO
    weights = trained(cfg)
    result = YOLO(str(weights)).val(data=str(artifacts(cfg) / 'configs/detector.yaml'), split=split,
                                   imgsz=cfg['DETECTOR']['imgsz'], device=cfg['DETECTOR']['device'],
                                   project=str(artifacts(cfg) / 'reports'), name='detector_'+split,
                                   exist_ok=True, plots=True)
    save(artifacts(cfg) / 'reports' / ('detector_'+split+'.json'),
         {k: float(v) for k, v in result.results_dict.items()})
'''
Path(WORK_ROOT, 'src', 'detector.py').write_text(SOURCE_FILES['detector.py'], encoding='utf-8')


**Módulo `environment.py`**


In [ ]:
SOURCE_FILES['environment.py'] = r'''"""Sondas en procesos separados; sin modificar el entorno instalado."""
import importlib.metadata
import json
import platform
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path
from common import artifacts, require, save


def probe(cfg, framework):
    report = {'python': sys.version, 'platform': platform.platform(), 'framework': framework}
    if framework == 'system':
        import psutil
        report['ram'] = dict(psutil.virtual_memory()._asdict())
        report['disk'] = dict(zip(('total', 'used', 'free'), shutil.disk_usage(cfg['WORK_ROOT'])))
        command = ['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free,driver_version', '--format=csv']
        try:
            report['nvidia_smi'] = subprocess.check_output(command, text=True, timeout=20)
        except (OSError, subprocess.SubprocessError) as exc:
            report['nvidia_smi'] = str(exc)
        report['internet'] = {}
        for host in ('https://pypi.org/simple/ultralytics/', 'https://github.com/ultralytics/assets',
                     'https://storage.googleapis.com/keras-applications/'):
            try:
                with urllib.request.urlopen(host, timeout=10) as response:
                    report['internet'][host] = response.status
            except Exception as exc:
                report['internet'][host] = str(exc)
        report['packages'] = {p: importlib.metadata.version(p) for p in
                              ('numpy', 'pillow', 'matplotlib', 'psutil', 'PyYAML')}
    elif framework == 'torch':
        import torch
        import ultralytics
        from ultralytics.cfg import DEFAULT_CFG_DICT
        from ultralytics.engine.trainer import BaseTrainer
        import inspect
        require(ultralytics.__version__ == cfg['ULTRALYTICS_VERSION'],
                'Ultralytics no coincide con la versión fijada. Instala la versión de configuración.')
        required = {'resume', 'save_period', 'time', 'device', 'epochs'}
        require(required <= DEFAULT_CFG_DICT.keys(), 'API Ultralytics incompatible.')
        report.update(torch=torch.__version__, ultralytics=ultralytics.__version__, cuda=torch.version.cuda,
                      gpu_count=torch.cuda.device_count(), api_defaults={k: DEFAULT_CFG_DICT[k] for k in required},
                      trainer_source_sha=__import__('hashlib').sha256(inspect.getsource(BaseTrainer).encode()).hexdigest())
        report['resume_implementation'] = inspect.getsource(BaseTrainer.check_resume)
        report['resume_training_implementation'] = inspect.getsource(BaseTrainer.resume_training)
        report['final_eval_implementation'] = inspect.getsource(BaseTrainer.final_eval)
        report['multi_gpu_implementation'] = inspect.getsource(BaseTrainer.train)
        require(torch.cuda.is_available(), 'PyTorch no detecta CUDA. Activa GPU en Kaggle.')
        report['gpus'] = [dict(index=i, name=torch.cuda.get_device_name(i),
                               memory_bytes=torch.cuda.get_device_properties(i).total_memory)
                          for i in range(torch.cuda.device_count())]
        device = int(cfg['DETECTOR']['device'])
        require(device < torch.cuda.device_count(), 'GPU configurada inexistente.')
        x = torch.randn((32, 32), device=f'cuda:{device}', requires_grad=True)
        x.square().mean().backward()
        torch.cuda.synchronize(device)
        report['gpu_operation_ok'] = True
    elif framework == 'tensorflow':
        import tensorflow as tf
        import keras
        gpu = tf.config.list_physical_devices('GPU')
        require(gpu, 'TensorFlow no detecta GPU. Revisa el entorno antes de entrenar.')
        tf.config.set_soft_device_placement(False)
        tf.config.set_visible_devices(gpu[int(cfg['REGRESSOR']['device'])], 'GPU')
        tf.config.experimental.set_memory_growth(tf.config.get_visible_devices('GPU')[0], True)
        with tf.device('/GPU:0'):
            x = tf.Variable(tf.ones((16, 16)))
            with tf.GradientTape() as tape:
                loss = tf.reduce_mean(tf.matmul(x, x))
            require(tape.gradient(loss, x) is not None, 'Falló operación GPU TensorFlow.')
        report.update(tensorflow=tf.__version__, keras=keras.__version__,
                      gpu_count=len(gpu), build=tf.sysconfig.get_build_info(), gpu_operation_ok=True)
    save(artifacts(cfg) / 'reports' / ('environment_' + framework + '.json'), report)
    print(json.dumps(report, indent=2, default=str))
'''
Path(WORK_ROOT, 'src', 'environment.py').write_text(SOURCE_FILES['environment.py'], encoding='utf-8')


**Módulo `evaluation.py`**


In [ ]:
SOURCE_FILES['evaluation.py'] = r'''"""Métricas continuas y cobertura explícita; ningún test durante fit."""
import collections
import math
from pathlib import Path
from common import (artifacts, contract, digest, load, require, save, sha, stage, write_csv)


def metrics(records):
    eligible = [r for r in records if r.get('true_bcs') is not None]
    predicted = [r for r in eligible if r.get('pred_bcs') is not None]
    errors = [r['pred_bcs']-r['true_bcs'] for r in predicted]
    result = {'eligible_images': len(eligible), 'predicted_images': len(predicted),
              'abstained_images': len(eligible)-len(predicted),
              'coverage': len(predicted)/len(eligible) if eligible else None,
              'mae_predicted': sum(abs(e) for e in errors)/len(errors) if errors else None,
              'rmse_predicted': math.sqrt(sum(e*e for e in errors)/len(errors)) if errors else None}
    for tolerance in (.25, .5):
        count = sum(abs(e) <= tolerance for e in errors)
        result[f'within_{tolerance}_all'] = count/len(eligible) if eligible else None
        result[f'within_{tolerance}_predicted'] = count/len(predicted) if predicted else None
    return result


def report(cfg, records, tag):
    root = artifacts(cfg)
    dest = root / 'reports' / tag
    dest.mkdir(parents=True, exist_ok=True)
    save(dest / 'metrics.json', metrics(records))
    write_csv(dest / 'predictions.csv', records,
              ['key', 'cow_id', 'true_bcs', 'pred_bcs', 'status'])
    bycow = collections.defaultdict(list)
    for r in records:
        if r['true_bcs'] is not None:
            bycow[r['cow_id']].append(r)
    cows = [{'cow_id': cow, **metrics(items)} for cow, items in sorted(bycow.items())]
    write_csv(dest / 'by_cow.csv', cows, ['cow_id', *metrics([])])
    save(dest / 'cow_macro.json', {k: sum(c[k] for c in cows if c[k] is not None)/sum(c[k] is not None for c in cows)
                                  if any(c[k] is not None for c in cows) else None
                                  for k in ('coverage', 'mae_predicted', 'rmse_predicted', 'within_0.5_all')})
    predicted = [r for r in records if r['pred_bcs'] is not None and r['true_bcs'] is not None]
    if predicted:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].scatter([r['true_bcs'] for r in predicted], [r['pred_bcs'] for r in predicted])
        axes[0].plot([1, 5], [1, 5], '--', color='gray')
        axes[0].set(xlabel='BCS referencia', ylabel='BCS predicho (continuo)')
        axes[1].hist([r['pred_bcs']-r['true_bcs'] for r in predicted])
        axes[1].set(xlabel='Error BCS', ylabel='Imágenes')
        fig.tight_layout()
        fig.savefig(dest / 'errors.png', dpi=150)
        plt.close(fig)
    print(tag, metrics(records))


def predict(cfg, split=None, external=False):
    from regressor import runtime, trained
    tf = runtime(cfg)
    root = artifacts(cfg)
    model_path = trained(cfg)
    model = tf.keras.models.load_model(model_path)
    samples = load(root / 'predicted' / ('external' if external else split) / 'predictions.json')
    from detector import trained as detector_trained
    settings = samples['settings']
    require(all(settings[k] == v for k, v in contract(cfg).items()), 'Recortes predichos de otros datos.')
    require(settings['detector_hash'] == sha(detector_trained(cfg)) and settings['inference'] == cfg['INFERENCE']
            and settings['size'] == cfg['REGRESSOR']['size'], 'Recortes automáticos incompatibles.')

    def predict_rows(data):
        output = []
        for r in data:
            value = None
            if r['crop'] is not None:
                im = tf.io.decode_png(tf.io.read_file(str(root / r['crop'])), channels=3)
                im = tf.ensure_shape(im, [cfg['REGRESSOR']['size'], cfg['REGRESSOR']['size'], 3])
                value = float(model(tf.cast(im[None], tf.float32), training=False).numpy()[0, 0])
                require(math.isfinite(value), 'Predicción BCS no finita.')
            output.append({'key': r['key'], 'cow_id': r['cow_id'], 'true_bcs': r['true_bcs'],
                           'pred_bcs': value, 'status': r['status']})
        return output

    predicted = predict_rows(samples['rows'])
    tag = 'inference_external' if external else 'end_to_end_'+split
    report(cfg, predicted, tag)
    if external:
        save(root / 'reports/inference_external/prediction.json', predicted)
        return
    gt = [{'key': r['key'], 'cow_id': r['cow_id'], 'true_bcs': r['bcs'], 'crop': r['crop'], 'status': 'ok'}
          for r in load(root / 'manifests/regression_samples.json') if r['split'] == split]
    report(cfg, predict_rows(gt), 'regressor_gt_'+split)
    # Baseline robusta calculada SOLO con entrenamiento, nunca con validación/test.
    import statistics
    baseline = statistics.median(r['bcs'] for r in load(root / 'manifests/regression_samples.json') if r['split'] == 'train')
    report(cfg, [{'key': r['key'], 'cow_id': r['cow_id'], 'true_bcs': r['true_bcs'],
                  'pred_bcs': baseline, 'status': 'train_median_baseline'} for r in gt], 'baseline_'+split)
    stage(cfg, 'evaluation_'+split,
          digest({**contract(cfg), 'model': sha(model_path), 'detector': settings['detector_hash'],
                  'inference': cfg['INFERENCE']}),
          [root / 'reports' / tag / 'metrics.json', root / 'reports' / ('regressor_gt_'+split) / 'metrics.json'])
'''
Path(WORK_ROOT, 'src', 'evaluation.py').write_text(SOURCE_FILES['evaluation.py'], encoding='utf-8')


**Módulo `regressor.py`**


In [ ]:
SOURCE_FILES['regressor.py'] = r'''"""Regresión continua MAE, dos fases y checkpoints transaccionales por época."""
import importlib.metadata
import json
import time
from pathlib import Path
from common import (artifacts, can_start, contract, copy_atomic, digest, load, require,
                    save, sha, stage, write_csv)


def runtime(cfg):
    import tensorflow as tf
    require(int(tf.__version__.split('.')[0]) >= 2, 'Se requiere TensorFlow 2 con Keras 3.')
    require(int(importlib.metadata.version('keras').split('.')[0]) >= 3, 'Se requiere Keras 3.')
    gpu = tf.config.list_physical_devices('GPU')
    require(gpu, 'TensorFlow no detecta GPU.')
    index = int(cfg['REGRESSOR']['device'])
    require(index < len(gpu), 'GPU del regresor inexistente.')
    # Una GPU: la sobrecarga de MirroredStrategy no se justifica sin benchmark.
    tf.config.set_visible_devices(gpu[index], 'GPU')
    tf.config.experimental.set_memory_growth(gpu[index], True)
    tf.keras.utils.set_random_seed(cfg['SEED'])
    return tf


def dataset(tf, cfg, selected, training=False):
    root = artifacts(cfg)
    size, batch = cfg['REGRESSOR']['size'], cfg['REGRESSOR']['batch']
    require(selected, 'Partición de regresión vacía.')
    paths = [str(root / r['crop']) for r in selected]
    values = [[float(r['bcs'])] for r in selected]
    ds = tf.data.Dataset.from_tensor_slices((paths, values))
    if training:
        ds = ds.shuffle(len(paths), seed=cfg['SEED'], reshuffle_each_iteration=True)

    def decode(path, value):
        im = tf.io.decode_png(tf.io.read_file(path), channels=3)
        im = tf.ensure_shape(im, [size, size, 3])
        return tf.cast(im, tf.float32), tf.cast(value, tf.float32)

    return ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)


def build(tf, cfg, weights=None):
    size = cfg['REGRESSOR']['size']
    inputs = tf.keras.Input((size, size, 3), name='rgb_0_255')
    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal', seed=cfg['SEED']),
        tf.keras.layers.RandomRotation(.03, seed=cfg['SEED']+1),
        tf.keras.layers.RandomZoom(.05, seed=cfg['SEED']+2),
    ], name='train_augmentation')
    base = tf.keras.applications.EfficientNetB0(include_top=False, weights=weights,
                                               input_shape=(size, size, 3))
    base.trainable = False
    features = base(augmentation(inputs), training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(features)
    x = tf.keras.layers.Dropout(.2, seed=cfg['SEED']+3)(x)
    output = tf.keras.layers.Dense(1, activation='linear', dtype='float32', name='bcs')(x)
    return tf.keras.Model(inputs, output, name='bcs_regressor')


def root_run(cfg, smoke=False):
    return artifacts(cfg) / ('smoke/regressor' if smoke else 'regressor') / cfg['EXPERIMENT_ID']


def phase_train(tf, cfg, phase, smoke=False, interrupt_after=None):
    root = artifacts(cfg)
    runroot = root_run(cfg, smoke)
    run = runroot / phase
    settings = cfg['REGRESSOR']
    epochs = cfg['SMOKE_EPOCHS'] if smoke else settings[phase+'_epochs']
    fp = digest({**contract(cfg), 'regressor': settings, 'phase': phase, 'epochs': epochs,
                 'smoke': smoke, 'seed': cfg['SEED'], 'tensorflow': tf.__version__,
                 'keras': importlib.metadata.version('keras')})
    state_path = run / 'state.json'
    state = load(state_path) if state_path.exists() else None
    if state:
        require(state['fingerprint'] == fp, 'Regresor incompatible: crea otro EXPERIMENT_ID.')
        require(sha(run / state['checkpoint']) == state['checkpoint_sha256'], 'Checkpoint Keras incompleto.')
        require(sha(run / state['best_checkpoint']) == state['best_checkpoint_sha256'], 'Mejor checkpoint Keras incompleto.')
        # Reconstruir aliases incluso si el proceso murió entre copias y commit del estado.
        copy_atomic(run / state['checkpoint'], run / 'last.keras')
        copy_atomic(run / state['best_checkpoint'], run / 'best.keras')
        write_csv(run / 'history.csv', state['history'])
        copy_atomic(run / 'history.csv', root / 'histories' / ('smoke_' if smoke else '') / ('regressor_'+phase+'.csv'))
        if state['status'] == 'complete':
            print('Fase ya completada:', phase)
            return state
    if not smoke:
        estimate_path = root / 'reports/regressor_smoke.json'
        require(estimate_path.is_file(), 'Ejecuta primero la prueba corta del regresor.')
        estimate = load(estimate_path)
        require(estimate['contract'] == contract(cfg) and estimate['settings'] == settings
                and estimate['seed'] == cfg['SEED'],
                'Prueba corta de regresión incompatible.')
        if not can_start(cfg, estimate['phases'][phase]['seconds_per_epoch']):
            print('Presupuesto insuficiente para otra época de', phase)
            return None
    require(can_start(cfg, 0), 'Se agotó el presupuesto útil de sesión.')
    run.mkdir(parents=True, exist_ok=True)
    selected = load(root / 'manifests/regression_samples.json')
    train = dataset(tf, cfg, [r for r in selected if r['split'] == 'train'], True)
    val = dataset(tf, cfg, [r for r in selected if r['split'] == 'val'])
    if state:
        model = tf.keras.models.load_model(run / state['checkpoint'])
        require(model.optimizer is not None, 'Modelo sin optimizador: no es una reanudación.')
        require(int(model.optimizer.iterations.numpy()) == state['optimizer_iterations'], 'Optimizador no restaurado.')
        require(model.output_shape[-1] == 1, 'Se esperaba regresión escalar.')
    else:
        if phase == 'head':
            model = build(tf, cfg, None if smoke else settings['weights'])
        else:
            head = runroot / 'head'
            headstate = load(head / 'state.json')
            require(headstate['status'] == 'complete', 'Finaliza head antes del ajuste fino.')
            require(headstate['contract'] == contract(cfg) and headstate['settings'] == settings
                    and headstate['seed'] == cfg['SEED'],
                    'La cabeza congelada pertenece a otra configuración o datos.')
            require(sha(head / headstate['best_checkpoint']) == headstate['best_checkpoint_sha256'],
                    'Checkpoint de la cabeza alterado.')
            model = tf.keras.models.load_model(head / headstate['best_checkpoint'])
            base = model.get_layer('efficientnetb0')
            base.trainable = True
            n = settings['fine_layers']
            require(0 < n <= len(base.layers), 'Número de capas de ajuste fino inválido.')
            for i, layer in enumerate(base.layers):
                layer.trainable = i >= len(base.layers)-n and not isinstance(layer, tf.keras.layers.BatchNormalization)
        model.compile(optimizer=tf.keras.optimizers.Adam(settings[phase+'_lr']),
                      loss=tf.keras.losses.MeanAbsoluteError(),
                      metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae')], jit_compile=False)
        state = {'fingerprint': fp, 'stage': 'regressor_'+phase, 'phase': phase,
                 'completed_epochs': 0, 'best_val_mae': None, 'wait': 0, 'history': [],
                 'status': 'running', 'contract': contract(cfg),
                 'settings': settings,
                 'seed': cfg['SEED'],
                 'config_path': 'configs/config.json', 'target_epochs': epochs}
    initial_epoch = state['completed_epochs']
    require(initial_epoch < epochs, 'La fase ya alcanzó su objetivo de épocas.')
    started = [None]

    class DurableEpoch(tf.keras.callbacks.Callback):
        def on_epoch_begin(self, epoch, logs=None):
            started[0] = time.monotonic()

        def on_epoch_end(self, epoch, logs=None):
            logs = {k: float(v) for k, v in (logs or {}).items()}
            require(all(__import__('math').isfinite(v) for v in logs.values()),
                    'Pérdida no finita: se conserva el checkpoint anterior.')
            number = epoch+1
            elapsed = time.monotonic()-started[0]
            filename = f'epoch_{number:04d}.keras'
            temp = run / f'epoch_{number:04d}.tmp.keras'
            self.model.save(temp)
            temp.replace(run / filename)
            improved = state['best_val_mae'] is None or logs['val_mae'] < state['best_val_mae']
            state['wait'] = 0 if improved else state['wait']+1
            if improved:
                state.update(best_val_mae=logs['val_mae'], best_epoch=number,
                             best_checkpoint=filename, best_checkpoint_sha256=sha(run / filename))
            state['history'].append({'epoch': number, **logs, 'seconds': elapsed})
            state.update(completed_epochs=number, checkpoint=filename, checkpoint_sha256=sha(run / filename),
                         optimizer_iterations=int(self.model.optimizer.iterations.numpy()),
                         learning_rate=float(tf.keras.backend.get_value(self.model.optimizer.learning_rate)),
                         trainable_layers=[l.name for l in self.model.layers if l.trainable],
                         base_trainable_layers=[l.name for l in self.model.get_layer('efficientnetb0').layers if l.trainable],
                         seconds_per_epoch=sum(h['seconds'] for h in state['history'])/len(state['history']))
            # Incluir coste de serialización al estimar la siguiente época.
            elapsed = time.monotonic()-started[0]
            state['history'][-1]['seconds'] = elapsed
            state['seconds_per_epoch'] = sum(h['seconds'] for h in state['history'])/len(state['history'])
            try:
                state['gpu_memory'] = tf.config.experimental.get_memory_info('GPU:0')
            except (ValueError, RuntimeError):
                state['gpu_memory'] = None
            done = number >= epochs or state['wait'] >= settings['patience']
            forced = interrupt_after is not None and number >= interrupt_after
            pause = not done and (forced or not can_start(cfg, max(h['seconds'] for h in state['history'])))
            state['status'] = 'complete' if done else 'paused' if pause else 'running'
            copy_atomic(run / filename, run / 'last.keras')
            copy_atomic(run / state['best_checkpoint'], run / 'best.keras')
            # Estado es el commit: apunta a archivos inmutables, nunca a un alias parcialmente actualizado.
            save(state_path, state)
            write_csv(run / 'history.csv', state['history'])
            copy_atomic(run / 'history.csv', root / 'histories' / ('smoke_' if smoke else '') / ('regressor_'+phase+'.csv'))
            keep = {filename, state['best_checkpoint'], f'epoch_{number-1:04d}.keras'}
            for path in run.glob('epoch_*.keras'):
                if path.name not in keep and '.tmp.' not in path.name:
                    path.unlink()
            if done or pause:
                self.model.stop_training = True

    model.fit(train, validation_data=val, initial_epoch=initial_epoch, epochs=epochs,
              callbacks=[DurableEpoch(), tf.keras.callbacks.TerminateOnNaN()], verbose=2)
    del model
    tf.keras.backend.clear_session()
    return load(state_path)


def train(cfg, smoke=False):
    tf = runtime(cfg)
    if smoke:
        require(cfg['SMOKE_EPOCHS'] >= 2, 'La prueba de reanudación necesita dos épocas.')
        phases = {}
        for phase in ('head', 'fine_tuning'):
            state = phase_train(tf, cfg, phase, True, interrupt_after=1)
            if state['status'] != 'complete':
                require(state['completed_epochs'] == 1, 'La prueba corta no pausó después de una época.')
                iterations = state['optimizer_iterations']
                state = phase_train(tf, cfg, phase, True)
                require(state['optimizer_iterations'] > iterations, 'No avanzó el optimizador al reanudar.')
            require(state['status'] == 'complete', 'Prueba corta pausada por presupuesto; repítela en otra sesión.')
            phases[phase] = {'seconds_per_epoch': state['seconds_per_epoch'],
                             'best_val_mae': state['best_val_mae'], 'gpu_memory': state.get('gpu_memory'),
                             'optimizer_restored': True}
        save(artifacts(cfg) / 'reports/regressor_smoke.json', {
            'contract': contract(cfg), 'settings': cfg['REGRESSOR'], 'phases': phases, 'seed': cfg['SEED'],
            'weights': 'random_for_technical_test',
            'estimated_full_minutes': sum(phases[p]['seconds_per_epoch']*cfg['REGRESSOR'][p+'_epochs']/60 for p in phases)})
        return
    phases = ('head', 'fine_tuning') if cfg['REGRESSOR_PHASE'] == 'all' else (cfg['REGRESSOR_PHASE'],)
    for phase in phases:
        state = phase_train(tf, cfg, phase)
        if state is None or state['status'] != 'complete':
            return
    select(cfg)


def select(cfg):
    root = root_run(cfg)
    candidates = []
    for phase in ('head', 'fine_tuning'):
        p = root / phase / 'state.json'
        if p.exists():
            s = load(p)
            require(s['contract'] == contract(cfg) and s['settings'] == cfg['REGRESSOR']
                    and s['seed'] == cfg['SEED'],
                    'Fase de otros datos/código/configuración.')
            if s['status'] == 'complete':
                candidates.append((s['best_val_mae'], phase, s))
    require(candidates, 'No hay fases completas para seleccionar.')
    metric, phase, state = min(candidates, key=lambda v: v[0])
    source = root / phase / state['best_checkpoint']
    copy_atomic(source, root / 'best.keras')
    lastphase = 'fine_tuning' if any(p == 'fine_tuning' for _, p, _ in candidates) else 'head'
    copy_atomic(root / lastphase / 'last.keras', root / 'last.keras')
    save(root / 'selected.json', {'contract': contract(cfg), 'settings': cfg['REGRESSOR'],
                                 'seed': cfg['SEED'],
                                 'phase': phase, 'best_val_mae': metric, 'sha256': sha(root / 'best.keras')})
    stage(cfg, 'regressor_trained', digest({**contract(cfg), 'settings': cfg['REGRESSOR']}),
          [root / 'selected.json', root / 'best.keras', root / 'last.keras'])


def trained(cfg):
    root = root_run(cfg)
    selected = load(root / 'selected.json')
    require(selected['contract'] == contract(cfg) and selected['settings'] == cfg['REGRESSOR']
            and selected['seed'] == cfg['SEED'],
            'Regresor de otros datos o configuración.')
    require(sha(root / 'best.keras') == selected['sha256'], 'Mejor modelo alterado.')
    # Bloquear una selección antigua mientras una fase posterior está a medio entrenar.
    for phase in ('head', 'fine_tuning'):
        p = root / phase / 'state.json'
        if p.exists():
            require(load(p)['status'] == 'complete', 'Existe una fase de regresión pendiente.')
    return root / 'best.keras'
'''
Path(WORK_ROOT, 'src', 'regressor.py').write_text(SOURCE_FILES['regressor.py'], encoding='utf-8')


**Módulo `run.py`**


In [ ]:
SOURCE_FILES['run.py'] = r'''"""Entrypoint subprocess: python src/run.py ACCION --config config.json."""
import argparse
import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path
from common import artifacts, contract, digest, load, require, restore, save, sha


def validate_inputs(cfg):
    root = artifacts(cfg)
    manifest = root / 'manifests/manifest.json'
    require(manifest.exists(), 'No hay manifiesto. Activa RUN_AUDIT.')
    m = load(manifest)
    state = load(root / 'state/audit.json')
    require(state['outputs']['manifests/manifest.json'] == sha(manifest), 'Manifiesto alterado.')
    for split, h in m['coco_hashes'].items():
        require(sha(Path(cfg['DATA_ROOT']) / split / '_annotations.coco.json') == h,
                'Cambió COCO: repite la auditoría en otro WORK_ROOT.')
    for row in m['rows']:
        require(sha(Path(cfg['DATA_ROOT']) / row['image']) == row['sha256'],
                'Cambió una imagen: repite auditoría en otro WORK_ROOT.')
    if (root / 'manifests/splits.json').exists():
        s = load(root / 'manifests/splits.json')
        require(s['split_hash'] == digest(s['assignments']) and
                s['assignments'] == {r['key']: r['split'] for r in m['rows']}, 'Particiones alteradas.')


def verify_prepared(cfg, action):
    names = ['partitions', 'yolo_export'] if action.startswith('detector') else ['partitions', 'regression_samples']
    if action in ('evaluate', 'lock-test', 'infer-regressor'):
        names = ['partitions', 'yolo_export', 'regression_samples']
    for name in names:
        p = artifacts(cfg) / 'state' / (name + '.json')
        require(p.is_file(), f'Falta etapa {name}; activa RUN_PREPARE_DATA.')
        state = load(p)
        require(all((artifacts(cfg) / f).is_file() and sha(artifacts(cfg) / f) == h
                    for f, h in state['outputs'].items()), f'Artefactos alterados en {name}.')
    if 'yolo_export' in names:
        from data import export_yolo
        export_yolo(cfg)  # Reconstruir YAML con rutas de working de esta sesión.


def lock_test(cfg):
    from detector import trained as detector_trained
    from regressor import trained as regressor_trained
    root = artifacts(cfg)
    require(cfg['FINAL_TEST_CONFIRMED'],
            'Cierra decisiones usando validación y configura FINAL_TEST_CONFIRMED=True para abrir test.')
    selection = {'contract': contract(cfg), 'detector': sha(detector_trained(cfg)),
                 'regressor': sha(regressor_trained(cfg)), 'inference': cfg['INFERENCE']}
    lock = root / 'state/test_selection.json'
    if lock.exists():
        require(load(lock) == selection, 'La selección cambió después de abrir test: no reutilices test para ajustar.')
    else:
        save(lock, selection)


def export(cfg):
    root = artifacts(cfg)
    root.mkdir(parents=True, exist_ok=True)
    inventory = {str(p.relative_to(root)): sha(p) for p in sorted(root.rglob('*'))
                 if p.is_file() and p.name != 'inventory.json' and '.tmp' not in p.name}
    save(root / 'inventory.json', inventory)
    target = Path(cfg['WORK_ROOT']).parent / (Path(cfg['WORK_ROOT']).name + '_artifacts.zip')
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as archive:
        for folder in (root, Path(cfg['WORK_ROOT']) / 'src'):
            for p in sorted(folder.rglob('*')):
                if p.is_file() and '__pycache__' not in p.parts and '.tmp' not in p.name:
                    archive.write(p, str(p.relative_to(Path(cfg['WORK_ROOT']))))
    print('Export descargable:', target, f'({target.stat().st_size/1024**2:.1f} MiB)')


def status(cfg):
    root = artifacts(cfg)
    names = {'audit': 'Auditoría completada y manifiesto creado', 'partitions': 'Particiones verificadas',
             'yolo_export': 'Exportación YOLO creada', 'regression_samples': 'Muestras de regresión creadas',
             'detector_trained': 'Detector entrenado', 'regressor_trained': 'Regresor entrenado',
             'evaluation_val': 'Evaluación de validación completada', 'evaluation_test': 'Evaluación de test completada'}
    for key, label in names.items():
        p = root / 'state' / (key + '.json')
        if key == 'detector_trained':
            p = root / 'detector' / cfg['EXPERIMENT_ID'] / 'state.json'
        elif key == 'regressor_trained':
            p = root / 'regressor' / cfg['EXPERIMENT_ID'] / 'selected.json'
            if p.exists():
                print('✓ ' + label + ' (selección por validación disponible)')
                continue
        print(('✓ ' if p.exists() and load(p).get('status') == 'complete' else '· ') + label)


def estimate(cfg):
    root = artifacts(cfg)
    stages, unknown = {}, []
    detector = root / 'reports/detector_smoke.json'
    state_path = root / 'detector' / cfg['EXPERIMENT_ID'] / 'state.json'
    state = load(state_path) if state_path.exists() else {}
    if state.get('status') == 'complete':
        stages['detector'] = 0
    elif detector.exists():
        stages['detector'] = load(detector)['seconds_per_epoch'] * max(0, cfg['DETECTOR']['epochs']-state.get('completed_epochs', 0))/60
    else:
        unknown.append('detector: falta prueba corta')
    reg = root / 'reports/regressor_smoke.json'
    for phase in ('head', 'fine_tuning'):
        p = root / 'regressor' / cfg['EXPERIMENT_ID'] / phase / 'state.json'
        s = load(p) if p.exists() else {}
        if s.get('status') == 'complete':
            stages[phase] = 0
        elif reg.exists():
            stages[phase] = load(reg)['phases'][phase]['seconds_per_epoch'] * max(0, cfg['REGRESSOR'][phase+'_epochs']-s.get('completed_epochs', 0))/60
        else:
            unknown.append(phase+': falta prueba corta')
    from common import remaining
    left = remaining(cfg)
    result = {'estimated_training_minutes': stages, 'unknown': unknown,
              'known_total_minutes': sum(stages.values()),
              'remaining_useful_minutes': None if left is None else max(0, left/60),
              'reserved_evaluation_and_export_minutes': cfg.get('SAFETY_MARGIN_MINUTES', 20),
              'note': 'Estimación orientativa; no incluye descarga/inicialización. No se reducen objetivos automáticamente.'}
    save(root / 'reports/time_projection.json', result)
    print(json.dumps(result, indent=2))


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('action')
    p.add_argument('--config', required=True)
    p.add_argument('--framework', choices=['system', 'torch', 'tensorflow'])
    args = p.parse_args()
    cfg = load(args.config)
    source_files = {p.name: p.read_text(encoding='utf-8') for p in Path(__file__).parent.glob('*.py')}
    require(hashlib.sha256(json.dumps(source_files, sort_keys=True).encode()).hexdigest() == cfg['CODE_HASH'],
            'Cambió el código generado después de configurar el notebook. Reejecuta la configuración.')
    require(cfg['SMOKE_EPOCHS'] >= 2, 'SMOKE_EPOCHS debe ser al menos 2.')
    require(cfg['REGRESSOR_PHASE'] in ('all', 'head', 'fine_tuning'), 'Fase de regresor inválida.')
    require(cfg['EVALUATION_SPLIT'] in ('val', 'test'), 'Partición de evaluación inválida.')
    require(str(cfg['DETECTOR']['device']).isdigit() and str(cfg['REGRESSOR']['device']).isdigit(),
            'Selecciona una GPU por etapa; DDP/MirroredStrategy requieren benchmark independiente.')
    root = artifacts(cfg)
    for folder in ('configs', 'manifests', 'state', 'reports', 'histories'):
        (root / folder).mkdir(parents=True, exist_ok=True)
    action = args.action
    if action not in ('restore', 'environment', 'inspect', 'audit', 'status', 'export', 'estimate'):
        validate_inputs(cfg)
    if action.startswith(('detector-', 'regressor-')) or action in ('evaluate', 'lock-test', 'infer-regressor'):
        verify_prepared(cfg, action)
    if action == 'restore':
        restore(cfg)
    elif action == 'environment':
        from environment import probe
        probe(cfg, args.framework)
    elif action in ('inspect', 'audit', 'partition', 'export-yolo', 'samples'):
        import data
        functions = {'inspect': data.inspect, 'audit': data.audit, 'partition': data.partition,
                     'export-yolo': data.export_yolo, 'samples': data.samples}
        functions[action](cfg)
    elif action in ('detector-smoke', 'detector-train'):
        from detector import train, run_dir
        smoke = action.endswith('smoke')
        if smoke:
            train(cfg, True, interrupt_after=1)
            s = load(run_dir(cfg, True) / 'state.json')
            if s['status'] != 'complete':
                train(cfg, True)
            s = load(run_dir(cfg, True) / 'state.json')
            require(s['status'] == 'complete' and s['completed_epochs'] >= 2,
                    'Prueba detector incompleta por presupuesto.')
        else:
            train(cfg)
    elif action in ('regressor-smoke', 'regressor-train'):
        from regressor import train
        train(cfg, action.endswith('smoke'))
    elif action == 'lock-test':
        lock_test(cfg)
    elif action in ('detector-evaluate', 'detector-crops'):
        import detector
        if cfg['EVALUATION_SPLIT'] == 'test':
            lock_test(cfg)
        if action == 'detector-evaluate':
            detector.evaluate(cfg, cfg['EVALUATION_SPLIT'])
        else:
            detector.detect(cfg, cfg['EVALUATION_SPLIT'])
    elif action == 'evaluate':
        if cfg['EVALUATION_SPLIT'] == 'test':
            lock_test(cfg)
        from evaluation import predict
        predict(cfg, cfg['EVALUATION_SPLIT'])
    elif action == 'infer-detector':
        from detector import detect
        require(cfg.get('INFERENCE_IMAGE'), 'Configura INFERENCE_IMAGE con una foto nueva.')
        detect(cfg, external=cfg['INFERENCE_IMAGE'])
    elif action == 'infer-regressor':
        from evaluation import predict
        predict(cfg, external=True)
    elif action == 'export':
        export(cfg)
    elif action == 'status':
        status(cfg)
    elif action == 'estimate':
        estimate(cfg)
    else:
        raise ValueError('Acción desconocida: ' + action)


if __name__ == '__main__':
    main()
'''
Path(WORK_ROOT, 'src', 'run.py').write_text(SOURCE_FILES['run.py'], encoding='utf-8')


In [ ]:
# Selección de dataset sin importar módulos de procesamiento ni frameworks.
input_root = Path('/kaggle/input')
print('Entradas disponibles:', sorted(p.name for p in input_root.iterdir()) if input_root.exists() else [])
if DATA_ROOT == '/kaggle/input/X':
    candidates = sorted({p.parent.parent for p in input_root.rglob('_annotations.coco.json')
                         if p.parent.name == 'train'
                         and (p.parent.parent/'valid/_annotations.coco.json').is_file()
                         and (p.parent.parent/'test/_annotations.coco.json').is_file()})
    print('Exports COCO candidatos:', [str(p) for p in candidates])
    if len(candidates) != 1:
        raise RuntimeError('Selecciona DATA_ROOT en la configuración. Hay varios exports o no se encontró ninguno; '
                           'adjunta el export COCO con imágenes. No se elegirá una entrada arbitraria.')
    DATA_ROOT = str(candidates[0])
if not Path(DATA_ROOT).is_dir():
    raise FileNotFoundError('DATA_ROOT no existe. Adjunta el export COCO con sus imágenes.')

CODE_HASH = hashlib.sha256(json.dumps(SOURCE_FILES, sort_keys=True).encode()).hexdigest()
keys = ['DATA_ROOT', 'WORK_ROOT', 'RESUME_ROOT', 'EXPERIMENT_ID', 'SEED',
        'SESSION_BUDGET_MINUTES', 'SAFETY_MARGIN_MINUTES', 'DEADLINE_UNIX',
        'SMOKE_EPOCHS', 'EVALUATION_SPLIT', 'FINAL_TEST_CONFIRMED', 'REGRESSOR_PHASE',
        'INFERENCE_IMAGE', 'ULTRALYTICS_VERSION', 'BOX_ROUNDING_TOLERANCE_PX',
        'DETECTOR', 'REGRESSOR', 'INFERENCE', 'CODE_HASH',
        'RUN_AUDIT', 'RUN_PREPARE_DATA', 'RUN_TRAIN_DETECTOR', 'RUN_TRAIN_REGRESSOR',
        'RUN_EVALUATION', 'RUN_DETECTOR_SMOKE', 'RUN_REGRESSOR_SMOKE', 'RUN_INFERENCE', 'RUN_EXPORT']
CFG = {key: globals()[key] for key in keys}
CONFIG_PATH = Path(WORK_ROOT, 'runtime_config.json')
CONFIG_PATH.write_text(json.dumps(CFG, indent=2), encoding='utf-8')

def run_stage(action, *extra):
    command = [sys.executable, '-u', str(Path(WORK_ROOT, 'src/run.py')), action,
               '--config', str(CONFIG_PATH), *extra]
    # Salida en vivo; cada proceso termina antes de iniciar el siguiente.
    env = os.environ.copy()
    env.update(MPLBACKEND='Agg', TF_CPP_MIN_LOG_LEVEL='2', WANDB_MODE='disabled')
    subprocess.run(command, check=True, env=env)

if SESSION_BUDGET_MINUTES is None:
    print('Presupuesto desconocido: consulta los límites actuales de tu cuenta; trabaja por etapas y persiste outputs.')


## 2. Comprobación del entorno y dependencias

Se conservan PyTorch y TensorFlow de Kaggle. No se instalan controladores CUDA ni se actualizan
globalmente frameworks. Si hay una versión diferente de Ultralytics, la celda se detiene para evitar cambiarla
silenciosamente: instala la versión fijada en una sesión limpia o valida explícitamente otra versión.
Sin Internet, adjunta los pesos y las ruedas necesarias; no se cambia a pesos aleatorios como alternativa.

Las sondas GPU se ejecutan al solicitar entrenamiento, pruebas cortas, evaluación o inferencia.
La auditoría y preparación pueden hacerse con acelerador desactivado.
La VRAM se reporta por dispositivo: dos T4 no forman una GPU con memoria compartida.
Esta entrega usa una GPU por etapa. DDP (`device=[0,1]`) y MirroredStrategy requieren un benchmark separado,
pruebas de reanudación y mejora medida antes de incorporarlos; no se activan por tener dos dispositivos.


In [ ]:
requirements = {'Pillow': 'Pillow>=10,<13', 'numpy': 'numpy>=1.26,<3',
                'matplotlib': 'matplotlib>=3.7,<4', 'psutil': 'psutil>=5.9,<8',
                'PyYAML': 'PyYAML>=6,<7', 'pandas': 'pandas>=2,<4'}
need_torch = RUN_DETECTOR_SMOKE or RUN_TRAIN_DETECTOR or RUN_EVALUATION or RUN_INFERENCE
if need_torch:
    requirements.update({'ultralytics': 'ultralytics=='+ULTRALYTICS_VERSION,
                         'ultralytics-thop': 'ultralytics-thop>=2,<3',
                         'scipy': 'scipy>=1.11,<2', 'requests': 'requests>=2,<3',
                         'polars': 'polars>=0.20,<2'})
missing = []
for package, spec in requirements.items():
    try:
        version = importlib.metadata.version(package)
        if package == 'ultralytics' and version != ULTRALYTICS_VERSION:
            raise RuntimeError(f'Ultralytics {version} ya instalado; se requiere {ULTRALYTICS_VERSION}. '
                               'Usa una sesión limpia y fija la versión antes de importar frameworks.')
    except importlib.metadata.PackageNotFoundError:
        missing.append(spec)
if missing:
    if not INSTALL_MISSING_DEPENDENCIES:
        raise RuntimeError('Faltan dependencias: '+', '.join(missing))
    # --no-deps evita que instalar Ultralytics reemplace torch, CUDA o numpy de Kaggle.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                    '--no-deps', *missing], check=True)
if need_torch:
    # Estos paquetes los aporta Kaggle; no reemplazarlos con una instalación automática.
    subprocess.run([sys.executable, '-c', 'import torch, torchvision, cv2, ultralytics'], check=True)
run_stage('restore')
artifact_root = Path(WORK_ROOT, 'artifacts')
(artifact_root/'configs').mkdir(parents=True, exist_ok=True)
(artifact_root/'configs/config.json').write_text(json.dumps(CFG, indent=2), encoding='utf-8')
# Registrar exactamente lo instalado; útil al preparar una sesión de reanudación.
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(artifact_root/'configs/packages.txt').write_text(freeze)
run_stage('environment', '--framework', 'system')
if RUN_DETECTOR_SMOKE or RUN_TRAIN_DETECTOR or RUN_EVALUATION or RUN_INFERENCE:
    run_stage('environment', '--framework', 'torch')
if RUN_REGRESSOR_SMOKE or RUN_TRAIN_REGRESSOR or RUN_EVALUATION or RUN_INFERENCE:
    run_stage('environment', '--framework', 'tensorflow')


## 3. Inspección del dataset COCO

Inspeccionar las categorías originales antes de exportar. El mapeo del detector es `0=cow`, `1=ass`;
el ID de categoría contenedora de Roboflow no se transforma en una clase entrenable.
Los IDs de imagen COCO son locales a cada partición; la clave de trazabilidad incluye el nombre de partición.


In [ ]:
run_stage('inspect')


## 4. Auditoría y manifiesto

Comprobar imágenes, dimensiones, cajas, etiquetas BCS, identidades y SHA-256 entre particiones.
Las fotos con ID/BCS ambiguos conservan supervisión de detección y se excluyen de regresión.
No se adivina cuál BCS corresponde a una grupa. Los desbordamientos de redondeo de hasta 0.02 píxeles
se limitan al borde y quedan documentados; errores mayores bloquean la preparación.
Las huellas se vuelven a verificar al ejecutar etapas posteriores, aunque `RUN_AUDIT=False`.


In [ ]:
if RUN_AUDIT:
    run_stage('audit')
summary_path = artifact_root/'reports/audit_summary.json'
if summary_path.exists():
    import pandas as pd
    display(pd.DataFrame(json.loads(summary_path.read_text())))
    print('Las proporciones se calculan por separado para detector y regresor después de las exclusiones.')


## 5. Particiones por vaca

Conservar `train/valid/test` originales después de verificar aislamiento de todas las identidades,
incluidas las etiquetas ambiguas. La validación no se reemplaza por test para reducir tiempos.
Guardar asignaciones y hash. Las correcciones futuras de datos requieren una nueva versión de trabajo.
SHA-256 no detecta todos los duplicados visualmente similares ni identidades mal anotadas: revisar muestras.


In [ ]:
if RUN_PREPARE_DATA:
    run_stage('partition')


## 6. Exportación a formato YOLO

Convertir `xywh` COCO a cajas normalizadas y guardar YAML de rutas de esta sesión.
Las fotos se copian a working para conservar una exportación portable; no se escriben etiquetas en input.
La exportación se reutiliza únicamente si sus archivos y huellas coinciden.


In [ ]:
if RUN_PREPARE_DATA:
    run_stage('export-yolo')


## 7. Entrenamiento o reanudación de YOLO26

Primero activa `RUN_DETECTOR_SMOKE`: dos épocas separadas del experimento final, con parada después
de la primera y recuperación del optimizador para la segunda. Se guarda tiempo por época y memoria pico.
Antes del entrenamiento largo revisa la estimación y consulta el tiempo restante de Kaggle.
Con presupuesto configurado se reserva el margen indicado y se pausa al completar una época.
Una interrupción abrupta puede perder la época en curso; ningún callback puede impedir que Kaggle termine la sesión.

Guardar `best.pt`, `last.pt`, periódicos y `resume.pt` antes de la limpieza del optimizador de Ultralytics.
El checkpoint recuperable preserva la época y estado de entrenamiento; `best.pt` se usa para inferencia.
Cambiar épocas después de finalizar no es una reanudación compatible en esta versión:
elige otro `EXPERIMENT_ID` y configura `DETECTOR['weights']` con los pesos anteriores para un nuevo experimento.
La paciencia de Ultralytics y el flujo aleatorio no se prometen idénticos al reanudar.

Referencias: [entrenamiento](https://docs.ultralytics.com/modes/train/),
[implementación fijada del trainer](https://github.com/ultralytics/ultralytics/blob/v8.4.0/ultralytics/engine/trainer.py).
La sonda guarda el código de reanudación y finalización de la versión realmente instalada.


In [ ]:
if RUN_DETECTOR_SMOKE:
    run_stage('detector-smoke')
estimate_path = artifact_root/'reports/detector_smoke.json'
if estimate_path.exists():
    print(estimate_path.read_text())
if RUN_TRAIN_DETECTOR:
    run_stage('estimate')
    run_stage('detector-train')


## 8. Generación de muestras de regresión

Usar cajas anotadas de grupa para entrenar. Crear recortes RGB con relleno 224×224, sin deformar.
El preprocesamiento será el mismo en entrenamiento, evaluación e inferencia. Los IDs solo sirven para
trazabilidad, particiones y reportes; no son entradas de la red.


In [ ]:
if RUN_PREPARE_DATA:
    run_stage('samples')
samples_path = artifact_root/'manifests/regression_samples.json'
if samples_path.exists():
    from IPython.display import display, Image as DisplayImage
    sample_rows = json.loads(samples_path.read_text())
    for row in sample_rows[:6]:
        print(row['split'], 'BCS:', row['bcs'])
        display(DisplayImage(filename=str(artifact_root/row['crop']), width=180))


## 9. Entrenamiento o reanudación de EfficientNet-B0 con MAE

Salida escalar lineal y pérdida MAE; no hay softmax, pesos de clase ni redondeo a categorías.
La red recibe RGB en 0..255 porque EfficientNet integra su escalado.
Solo entrenamiento usa aumentos; BatchNormalization permanece congelada durante ajuste fino.

Activar primero `RUN_REGRESSOR_SMOKE`: usa pesos aleatorios únicamente para comprobar las dos fases,
guardar/cargar modelos completos y verificar que el contador del optimizador continúa tras una pausa.
Sus métricas no son resultados científicos ni seleccionan el modelo final.
El entrenamiento final usa ImageNet o los pesos locales configurados.

Cada época guarda un modelo completo, estado JSON e historial CSV. `best.keras` se selecciona por
MAE de validación; `last.keras` refleja la última época completa. El estado restaura mejor métrica,
paciencia y optimizador; se conservan checkpoints inmutables para recuperación ante escrituras interrumpidas.
Entre cabeza y ajuste fino se recompila con un optimizador nuevo: es un cambio de fase deliberado.
La mejor fase se selecciona por validación. No se garantiza continuidad bit a bit de RNG o del cargador.

Referencia: [EfficientNet de Keras](https://keras.io/api/applications/efficientnet/).


In [ ]:
if RUN_REGRESSOR_SMOKE:
    run_stage('regressor-smoke')
estimate_path = artifact_root/'reports/regressor_smoke.json'
if estimate_path.exists():
    print(estimate_path.read_text())
if RUN_TRAIN_REGRESSOR:
    run_stage('estimate')
    run_stage('regressor-train')


## 10. Evaluación

Por defecto evaluar validación. Cerrar hiperparámetros, fase y umbrales antes de configurar
`EVALUATION_SPLIT="test"` y `FINAL_TEST_CONFIRMED=True`. Se guarda una selección con hashes para
detectar cambios después de abrir test. No es un mecanismo para impedir que alguien cree otro proyecto;
la independencia de test también requiere disciplina experimental.

Reportar detección (precisión, recall, mAP), regresión con cajas anotadas y pipeline con cajas predichas.
La asociación automática exige un único par vaca/grupa con centro contenido y cobertura mínima configurada.
La ausencia o ambigüedad produce abstención sin consultar ID, BCS ni cajas reales.
MAE y RMSE del sistema se calculan sobre predicciones disponibles y se muestran junto con cobertura;
las tolerancias globales incluyen abstenciones como fallos. Agregar reportes por vaca y promedio macro.
Comparar también con la mediana BCS de entrenamiento. No recortar predicciones al rango ni redondearlas.


In [ ]:
if RUN_EVALUATION:
    run_stage('detector-evaluate')
    run_stage('detector-crops')
    run_stage('evaluate')
    for tag in ('regressor_gt_', 'end_to_end_', 'baseline_'):
        print(tag+EVALUATION_SPLIT)
        print((artifact_root/'reports'/(tag+EVALUATION_SPLIT)/'metrics.json').read_text())


## 11. Inferencia y visualización

Configurar `INFERENCE_IMAGE` con una foto nueva. Ejecutar detector y regresor en procesos sucesivos.
La salida es un BCS continuo o una abstención con motivo; los scores de detección no son confianza BCS.
También mostrar ejemplos de la última evaluación cuando estén disponibles.


In [ ]:
if RUN_INFERENCE:
    run_stage('infer-detector')
    run_stage('infer-regressor')
    print((artifact_root/'reports/inference_external/prediction.json').read_text())
from IPython.display import display, Image as DisplayImage
preview_tag = 'external' if RUN_INFERENCE else EVALUATION_SPLIT
for path in sorted((artifact_root/'predicted'/preview_tag).glob('*_boxes.jpg'))[:4]:
    display(DisplayImage(filename=str(path), width=420))
predictions_path = artifact_root/'predicted'/preview_tag/'predictions.json'
if predictions_path.exists():
    for row in json.loads(predictions_path.read_text())['rows'][:4]:
        print('Asociación:', row['status'])
        if row['crop']:
            display(DisplayImage(filename=str(artifact_root/row['crop']), width=180))
for tag in ('regressor_gt_', 'end_to_end_'):
    path = artifact_root/'reports'/(tag+EVALUATION_SPLIT)/'errors.png'
    if path.exists():
        display(DisplayImage(filename=str(path), width=720))


## 12. Exportación de artefactos y continuación entre sesiones

1. Ejecutar esta sección antes de terminar. Revisar estados; una etapa pausada no está completada.
2. En **Save Version**, usar una modalidad que guarde outputs y comprobar la opción correspondiente.
   Para guardar el estado interactivo sin repetir entrenamiento, revisar las opciones avanzadas de Quick Save
   y activar el guardado de outputs si está disponible. **Save & Run All** vuelve a ejecutar el notebook:
   revisar banderas antes de usarlo. La interfaz puede cambiar; verificar el resultado guardado.
3. Abrir los outputs de la versión y comprobar que aparecen `artifacts/state`, checkpoints e inventario.
   Descargar también `bcs_project_artifacts.zip` desde el panel de archivos de Kaggle.
4. En una sesión nueva, adjuntar los outputs de esa versión como entrada, además del dataset original.
   Apuntar `RESUME_ROOT` a la carpeta que contiene `artifacts` o a `artifacts` directamente.
   Si solo se conserva el ZIP, extraerlo a working y usar esa carpeta como `RESUME_ROOT`.
5. Mantener `EXPERIMENT_ID` y parámetros; configurar únicamente las etapas pendientes y el presupuesto
   disponible de la nueva sesión. El notebook valida el inventario y copia a working sin alterar input.
   No basta adjuntar los pesos: la reanudación requiere estados, manifiestos y configuración compatibles.

Guardar en `/kaggle/working` o el autoguardado del código **no garantiza** conservar los outputs tras perder
la sesión. Verificar la versión persistida antes de cerrar. Sin presupuesto conocido, exportar después de cada etapa.
El ZIP incluye datos preparados y reportes privados: mantener privados el notebook, dataset y outputs.
Los archivos fuente publicados en Git deben permanecer sin outputs ni datos de animales.

Referencia: [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks).


In [ ]:
run_stage('status')
if RUN_EXPORT:
    run_stage('export')
    print('Descargar desde el panel de archivos:', Path(WORK_ROOT).parent/(Path(WORK_ROOT).name+'_artifacts.zip'))
